# 12 — Multimodal Prompt Engineering

## Scenario
Northstar automatically extracts data from invoice images submitted by users. 
However, users often provide contradictory text (e.g., typing "Here is my invoice for $800", but attaching an image that says "$500").

**The Problem:** We cannot trust the user's text, nor can we assume the OCR/Model will always perfectly read the image. If there is a contradiction, the model should *not* guess or compromise; it must flag the uncertainty so we can escalate to a human.


In [ ]:
from pathlib import Path
import sys

COURSE_DIR = Path.cwd()
sys.path.insert(0, str(COURSE_DIR))
from northstar.runtime import get_client
from lab12 import CASES, InvoiceExtraction, build_requests, claimed_amount, route


def show_request(request):
    print("SYSTEM:\n", request.system)
    for message in request.messages:
        if message.text:
            print(f"{message.role.upper()}:\n{message.text}")
        for part in message.parts:
            print(f"{message.role.upper()} PART:", part)

client = get_client(COURSE_DIR / "fixtures/replays.json")


## Step 1: The Multimodal Contradiction

We will pass the image (showing $500) AND a user prompt (claiming $800) to the model simultaneously. 
We use Pydantic to validate a typed evidence proposal rather than accepting an unconstrained string. The schema does not prove that the extracted values are correct.


In [ ]:
request = next(r for r in build_requests() if r.case_id == "i12/claim-800")
show_request(request)
print("IMAGE PART:", request.messages[0].parts[0])
response = client.generate(request)
print("RECORDED RESPONSE:", response.text)
extraction = InvoiceExtraction.model_validate_json(response.text)
print("PARSED:", extraction)
assert route(extraction, claimed_amount(CASES[0]["message"])) == "human_review"


## Step 2: Reconcile the user claim with extracted evidence

The application also escalates a mismatch even if a replayed model response incorrectly says there is no contradiction.


In [ ]:
for case in CASES[1:]:
    request = next(r for r in build_requests() if r.case_id == f"i12/{case['id']}")
    show_request(request)
    response = client.generate(request)
    extraction = InvoiceExtraction.model_validate_json(response.text)
    decision = route(extraction, claimed_amount(case["message"]))
    print("RECORDED RESPONSE:", response.text)
    print("PARSED:", extraction, "DECISION:", decision)
    assert decision == case["expected"]


## Conclusion

Native multimodality and typed outputs make extraction inspectable, but they do not make it correct. The application compares the user claim with the extracted value, validates source locators, and routes low-confidence or conflicting cases to review. An accepted extraction remains data—it does not authorize payment or refund execution.


## Takeaway
The recorded invoice run routes the $800 contradiction, missing-claim case, and low-confidence extraction to human review. The matching high-confidence $500 case reaches `accepted_extraction`, which is deliberately not an authorization to pay.


## References
- [Core Concepts & Workflow](README.md#core-concepts--workflow)
- [Deep dive](README.md#deep-dive)
- [Lab walkthrough](README.md#lab-walkthrough)
